[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/kane_mele/kane_mele.ipynb)

# Edge States of a Graphene Flake

On a graphene flake with spin-orbit coupling, an electron's spin decides which way it runs along the edge. The coupling opens a gap in the energies of the flake's interior, and the states inside that gap live on the edge only, the two spins circling it in opposite senses. A difference between the two kinds of site, as in boron nitride, opens a plain gap instead, and once it outweighs the spin-orbit coupling the edge states are gone. This notebook builds the flake's Hamiltonian as a sparse extensor, follows its levels as that difference grows, and launches an electron at the edge.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from numga.gatype import GAType
from numga.sparse import SparseExtensor
from examples.animation import save_animation
from examples.quantum.kane_mele import core, render

np.set_printoptions(precision=3, suppress=True)

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Even = ga.gatype.even()
# The spinors with spin up along z, and those with spin down.
Up = ga.gatype(ga.subspace("1 xy"))
Down = ga.gatype(ga.subspace("yz zx"))

# Energies in units of the hop between neighbours, lengths in lattice constants, times in the inverse of
# the hop's energy. The flake's size, its edge to centre in half lattice constants, and the outer share of
# it counted as its rim.
SIZE = 12
RIM = 0.3
# The spin-orbit coupling, and the masses swept past it.
SPIN_ORBIT = 0.1
MASSES = np.linspace(0.0, 0.8, 41)
# The levels nearest zero followed through the sweep, among the spin-up states, which hold each energy twice.
LEVELS = 32
# The masses whose states nearest zero are drawn: below the spin-orbit coupling's 3 * 3**0.5 times, and above.
CASES = np.array([0.0, 0.8])
# The states inside the gap without a mass, each energy four times, and the electron's spinor, spin along
# x, half up and half down.
IN_GAP = 32
SPINOR = (1 - mv.zx) * 0.5**0.5
# The time for the edge states to run once around the flake, and the frames over it.
LAP = 50.0
FRAMES = 50

# A hexagonal flake with armchair edges: its atoms, their sublattices, and their neighbours and second
# neighbours.
flake = core.flake(SIZE, RIM)

## 1. The Hamiltonian

An electron's state on each atom is a spinor, an even multivector. `hamiltonian` couples the atoms through multivectors: minus one for each hop to a neighbour; `mv.xy` times the spin-orbit strength for each hop to a second neighbour, signed by the sense the path turns through the atom between; and the mass on each atom, plus on one sublattice and minus on the other. Every coupling multiplies its spinor from the left by a scalar or by `mv.xy`, which leaves the spin along z as it is. The spin-up spinors `Up`, the even multivectors of the xy plane, stay spin-up, and the spin-down ones `Down`, the planes through z, stay spin-down: `energy * Up` and `energy * Down` are maps of each sector into itself.

In the notation of Kane and Mele, with a column of two complex amplitudes on each atom, the Hamiltonian reads as $H = -\sum_{\langle ij\rangle} c_i^\dagger c_j + i\lambda\sum_{\langle\langle ij\rangle\rangle}\nu_{ij}\, c_i^\dagger s_z c_j + m\sum_i \xi_i\, c_i^\dagger c_i$.

In [ ]:
def hamiltonian(flake: core.Flake, spin_orbit: float, mass: float) -> SparseExtensor:
    """The flake's Hamiltonian, coupling its atoms through multivectors."""
    atoms = flake.sublattice.shape[0]
    # Minus one for each hop between neighbours.
    hop = SparseExtensor(mv.scalar(-np.ones((len(flake.first), 1))), *flake.first.T, (atoms, atoms))   # [atoms, atoms] Scalar
    # For each hop to a second neighbour, the xy plane, signed by the sense the path turns.
    turn = SparseExtensor(mv.xy * flake.senses, *flake.second.T, (atoms, atoms))   # [atoms, atoms] Bivector
    # Plus one on the atoms of one sublattice, minus one on the other.
    stagger = SparseExtensor.from_diagonal(mv.scalar(flake.sublattice[:, None]))   # [atoms, atoms] Scalar
    return hop + turn * spin_orbit + stagger * mass                            # [atoms, atoms] Up


energy = hamiltonian(flake, SPIN_ORBIT, 0.0)                                   # [atoms, atoms] Up
up_sector, down_sector = (energy * Up).gatype, (energy * Down).gatype

In [ ]:
print("couplings:", energy.cells.gatype)
print("spin up:  ", up_sector)
print("spin down:", down_sector)

## 2. The levels as the mass grows

`levels` finds the energies nearest zero within one sector, and their states, with the sparse eigensolver; `rim_share` says how much of each state lies on the flake's rim. Without a mass the gap is full of levels on the rim, evenly spaced: standing waves running around the edge. As the mass grows past $3\sqrt3$ times the spin-orbit coupling, dashed, the interior's gap closes and reopens, empty.

In [ ]:
def levels(energy: SparseExtensor, sector: GAType, count: int) -> tuple[Scalar, Even]:
    """The count energies nearest zero within a sector of spinors, and their states, orthonormal."""
    atoms = energy.shape[0]
    unit = SparseExtensor.from_diagonal(mv.scalar(np.ones((atoms, 1))))      # [atoms, atoms] Scalar
    return (energy * sector).eigh(unit * sector, count)                       # [count] Scalar, [count, atoms] sector


def rim_share(flake: core.Flake, states: Even) -> Scalar:
    """How much of each state lies on the flake's rim."""
    density = states.scalar_norm_squared()                                     # [..., atoms] Scalar
    return (density * flake.rim).sum(axis=-1) / density.sum(axis=-1)          # [...] Scalar


spectra = [levels(hamiltonian(flake, SPIN_ORBIT, mass), Up, LEVELS) for mass in MASSES]
energies = stack([level for level, _ in spectra])                              # [masses, levels] Scalar
shares = stack([rim_share(flake, states) for _, states in spectra])            # [masses, levels] Scalar
render.draw_levels(MASSES, energies, shares, 3 * 3**0.5 * SPIN_ORBIT);

## 3. A state nearest zero

Without a mass the state nearest zero is a ring around the edge. With the mass well past the spin-orbit coupling it is a wave over the interior.

In [ ]:
nearest = stack([levels(hamiltonian(flake, SPIN_ORBIT, mass), Up, 2)[1][0] for mass in CASES])   # [cases, atoms] Up
densities = nearest.scalar_norm_squared()                                      # [cases, atoms] Scalar
render.draw_states(flake, densities, [f"mass {mass}" for mass in CASES]);

## 4. The two spins on the edge

`spread` starts an electron on the atom in the middle of the edge facing x, with its spin along x, half up and half down, and keeps its part inside the gap. Time turns each state on its right by `(mv.xy * (-energy * time)).exp()`, at the rate of its energy, and `psi >> mv.z` is the spin density: as long as the electron's density and pointing along its spin, drawn red where it is up and blue where it is down. The spin-up half runs clockwise around the flake and the spin-down half counterclockwise.

In the notation of the Schrödinger equation, each state's turn on the right reads as its phase $e^{-iEt}$.

In [ ]:
def spread(states: Even, energies: Scalar, start: int, spinor: Even, times: np.ndarray):
    """An electron started on one atom with the given spinor, keeping only its part in the given states, at
    each time: its spin density `psi >> mv.z`, as long as the electron's density and pointing along its spin."""
    weights = states[:, start].reverse().scalar_product(spinor)                # [states] Scalar
    for time in times:
        # Each state turns on its right at the rate of its energy.
        turned = weights * (mv.xy * (-energies * time)).exp()                  # [states] Even
        yield (states * turned[:, None]).sum(axis=0) >> mv.z                   # [atoms] Vector


in_gap, edge_states = levels(energy, Even, IN_GAP)                             # [states] Scalar, [states, atoms] Even
times = np.linspace(0.0, LAP, FRAMES + 1)
spins = stack(list(spread(edge_states, in_gap, flake.start, SPINOR, times)))   # [frames + 1, atoms] Vector
display(Image(filename=save_animation(render.animate_spin(flake, spins), "kane_mele_helical", 120)))

In [ ]:
# checks
# The Hamiltonian is its own reverse: every coupling's reverse couples the other way.
np.testing.assert_array_equal((~energy).cells.kernel, energy.cells.kernel)
np.testing.assert_array_equal((~energy).columns, energy.columns)
# Without a mass the levels inside the bulk gap lie on the rim; with the mass well past the spin-orbit
# coupling, the gap is empty.
half = np.abs(3 * 3**0.5 * SPIN_ORBIT - MASSES)
inside = np.abs(energies.to_array()[0]) < 0.9 * half[0]
assert inside.sum() >= 8 and np.all(shares.to_array()[0, inside] > 0.9)
assert np.abs(energies.to_array()[-1]).min() > 0.5 * half[-1]
# The electron's density keeps its total, and stays on the rim.
density = spins.norm()                                                         # [frames + 1, atoms] Scalar
totals = density.sum(axis=-1).to_array()
np.testing.assert_allclose(totals, totals[0], rtol=1e-9)
np.testing.assert_array_less(0.9 * totals, (density * flake.rim).sum(axis=-1).to_array())
# A quarter lap on, the spin-up half has turned clockwise and the spin-down half counterclockwise.
along = spins[FRAMES // 4] | mv.z                                              # [atoms] Scalar
up, down = (density[FRAMES // 4] + along) * 0.5, (density[FRAMES // 4] - along) * 0.5
start = flake.positions[flake.start].normalized()                             # [] Vector
centres = [((half * flake.positions).sum(axis=0) / half.sum(axis=0)).normalized() for half in (up, down)]
sines = [(mv.xy | (centre ^ start)).to_array().item() for centre in centres]
assert sines[0] < -0.5 and sines[1] > 0.5